In [7]:
from pathlib import Path
from getpass import getpass
import os
import pandas as pd
import numpy as np
import psycopg

proyecto = Path.cwd()
if proyecto.name == "notebooks":
    proyecto = proyecto.parent

consulta = """
SELECT o.observed_at, o.pm25, o.pm10, o.so2, o.no2, o.co, o.o3,
       o.temp, o.pres, o.dewp, o.rain, o.wd, o.wspm
FROM observations o
JOIN stations s ON o.station_id = s.id
WHERE s.name = %s
ORDER BY o.observed_at
"""

clave = os.getenv("BEIJING_DB_PASSWORD")
if not clave:
    clave = getpass("Contraseña de PostgreSQL: ")

with psycopg.connect(host="127.0.0.1", port=5432, dbname="beijing_pm25",
                    user="beijing_app", password=clave) as conexion:
    with conexion.cursor() as cursor:
        cursor.execute(consulta, ("Wanliu",))
        filas = cursor.fetchall()
        columnas = [col.name for col in cursor.description]

del clave
datos = pd.DataFrame(filas, columns=columnas)
if datos.empty:
    raise ValueError("No hay registros de Wanliu en PostgreSQL")
print("Registros de Wanliu:", len(datos))
display(datos.head())

Registros de Wanliu: 35064


,observed_at,pm25,pm10,so2,no2,co,o3,temp,pres,dewp,rain,wd,wspm
0,2013-03-01 00:00:00,8.0,8.0,6.0,28.0,400.0,52.0,-0.7,1023.0,-18.8,0.0,NNW,4.4
1,2013-03-01 01:00:00,9.0,9.0,6.0,28.0,400.0,50.0,-1.1,1023.2,-18.2,0.0,N,4.7
2,2013-03-01 02:00:00,3.0,6.0,NaN,19.0,400.0,55.0,-1.1,1023.5,-18.2,0.0,NNW,5.6
3,2013-03-01 03:00:00,11.0,30.0,8.0,14.0,NaN,NaN,-1.4,1024.5,-19.4,0.0,NW,3.1
4,2013-03-01 04:00:00,3.0,13.0,9.0,NaN,300.0,54.0,-2.0,1025.2,-19.5,0.0,N,2.0


In [8]:
datos["observed_at"] = pd.to_datetime(datos["observed_at"])
datos = datos.sort_values("observed_at").reset_index(drop=True)

if datos["observed_at"].duplicated().any():
    raise ValueError("Hay fechas duplicadas")

saltos = datos["observed_at"].diff().dropna() != pd.Timedelta(hours=1)
if saltos.any():
    raise ValueError("Se encontraron horas faltantes en el calendario")

numericas = ["pm25", "pm10", "so2", "no2", "co", "o3", "temp", "pres", "dewp", "rain", "wspm"]
for columna in numericas:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")

datos["wd"] = datos["wd"].fillna("Desconocido")

print("Saltos horarios:", saltos.sum())
print("PM2.5 faltante:", datos["pm25"].isna().sum())
display((datos[numericas].isna().mean() * 100).round(2).sort_values(ascending=False))

Saltos horarios: 0
PM2.5 faltante: 382


o3      6.01
co      5.17
no2     3.05
so2     1.64
pm25    1.09
pm10    0.81
temp    0.06
pres    0.06
dewp    0.06
rain    0.06
wspm    0.04
dtype: float64

In [9]:
preparado = datos.copy()
preparado["hora"] = preparado["observed_at"].dt.hour
preparado["mes"] = preparado["observed_at"].dt.month
preparado["dia_semana"] = preparado["observed_at"].dt.dayofweek

for horas in list(range(24)) + [24, 36, 47]:
    preparado[f"pm25_hist_{horas:02d}"] = preparado["pm25"].shift(horas)

for ventana in [6, 12, 24, 48]:
    preparado[f"pm25_media_{ventana}"] = preparado["pm25"].rolling(ventana, min_periods=1).mean()
    preparado[f"pm25_cuenta_{ventana}"] = preparado["pm25"].rolling(ventana, min_periods=1).count()

for ventana in [24, 48]:
    preparado[f"pm25_std_{ventana}"] = preparado["pm25"].rolling(ventana, min_periods=2).std()

for variable in ["temp", "pm10"]:
    for horas in [1, 24]:
        preparado[f"{variable}_lag_{horas}"] = preparado[variable].shift(horas)

print("Variables históricas creadas:")
display(preparado[["observed_at", "pm25", "pm25_hist_01", "pm25_media_24", "pm25_media_48"]].tail())

Variables históricas creadas:


C:\Users\PC\AppData\Local\Temp\ipykernel_20352\3226121297.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  preparado["dia_semana"] = preparado["observed_at"].dt.dayofweek


,observed_at,pm25,pm25_hist_01,pm25_media_24,pm25_media_48
35059,2017-02-28 19:00:00,11.0,6.0,16.958333,39.583333
35060,2017-02-28 20:00:00,15.0,11.0,15.583333,39.229167
35061,2017-02-28 21:00:00,13.0,15.0,13.416667,38.645833
35062,2017-02-28 22:00:00,12.0,13.0,11.333333,37.875000
35063,2017-02-28 23:00:00,7.0,12.0,9.958333,36.979167


In [10]:
for horas in range(1, 25):
    preparado[f"y_{horas:02d}"] = preparado["pm25"].shift(-horas)

historia_valida = preparado["observed_at"].shift(47) == preparado["observed_at"] - pd.Timedelta(hours=47)
futuro_valido = preparado["observed_at"].shift(-24) == preparado["observed_at"] + pd.Timedelta(hours=24)
columnas_objetivo = [f"y_{horas:02d}" for horas in range(1, 25)]
objetivos_completos = preparado[columnas_objetivo].notna().all(axis=1)

valido = historia_valida & futuro_valido & preparado["pm25"].notna() & objetivos_completos
preparado = preparado[valido].copy()

print("Orígenes válidos para 24 pronósticos:", len(preparado))
print("Ejemplos de aprendizaje posibles:", len(preparado) * 24)
display(preparado[["observed_at", "pm25", "y_01", "y_12", "y_24"]].head())

Orígenes válidos para 24 pronósticos: 31125
Ejemplos de aprendizaje posibles: 747000


,observed_at,pm25,y_01,y_12,y_24
47,2013-03-02 23:00:00,84.0,88.0,97.0,17.0
48,2013-03-03 00:00:00,88.0,104.0,101.0,19.0
49,2013-03-03 01:00:00,104.0,106.0,107.0,44.0
50,2013-03-03 02:00:00,106.0,96.0,84.0,42.0
51,2013-03-03 03:00:00,96.0,89.0,43.0,18.0


In [11]:
fechas = datos["observed_at"].sort_values().reset_index(drop=True)
corte_train = fechas.iloc[int(len(fechas) * 0.70)]
corte_val = fechas.iloc[int(len(fechas) * 0.85)]

fin_prediccion = preparado["observed_at"] + pd.Timedelta(hours=24)

preparado["conjunto"] = "excluido"
preparado.loc[fin_prediccion < corte_train, "conjunto"] = "train"
preparado.loc[(preparado["observed_at"] >= corte_train) & (fin_prediccion < corte_val), "conjunto"] = "val"
preparado.loc[preparado["observed_at"] >= corte_val, "conjunto"] = "test"
preparado = preparado[preparado["conjunto"] != "excluido"].copy()
preparado = preparado.reset_index(drop=True)

print("Corte entrenamiento:", corte_train)
print("Corte validación:", corte_val)
print("Ejemplos por conjunto:")
print(preparado["conjunto"].value_counts())

assert (preparado.loc[preparado["conjunto"] == "train", "observed_at"] + pd.Timedelta(hours=24) < corte_train).all()
assert (preparado.loc[preparado["conjunto"] == "val", "observed_at"] + pd.Timedelta(hours=24) < corte_val).all()

Corte entrenamiento: 2015-12-18 16:00:00
Corte validación: 2016-07-24 20:00:00
Ejemplos por conjunto:
conjunto
train    21956
test      4641
val       4511
Name: count, dtype: int64


In [12]:
salida = proyecto / "data" / "processed" / "wanliu_24h_preparado.csv.gz"
salida.parent.mkdir(parents=True, exist_ok=True)
preparado.to_csv(salida, index=False, compression="gzip")

print("Archivo guardado:", salida)
print("Filas:", len(preparado))
print("Columnas:", len(preparado.columns))

Archivo guardado: c:\Users\PC\Proyectos\beijing-pm25-crisp-mlq\data\processed\wanliu_24h_preparado.csv.gz
Filas: 31108
Columnas: 82
